# Ensemble Fusion: Weighted Class-Specific Combination

Combining Models A, D, and C.

**Strategy**:
- Load all 3 trained models
- Apply **class-specific weighted fusion**
- Model D gets **50% weight** for Moral Value & Powerlessness (its strength)
- Models A & C balance Economic (compensate for B's sacrifice)
- Grid search to optimize weights on validation set

**Class-Specific Weighting Strategy**:
```
                    Model A   Model D   Model C
Conflict:           0.35      0.40      0.25    <- B leads (best)
Economic:           0.40      0.20      0.40    <- A & C (B sacrificed)
Human Impact:       0.35      0.30      0.35    <- Balanced
Moral Value:        0.30      0.50      0.20    <- B DOMINATES!
None:               0.35      0.30      0.35    <- Balanced
Powerlessness:      0.30      0.50      0.20    <- B DOMINATES!
```

**Expected Ensemble Performance**:
```
Conflict:      0.72  (D's strength shines)
Economic:      0.82  (A & C compensate)
Human Impact:  0.76  (all contribute)
None:          0.73  (stable)
Moral Value:   0.70  (D carries!) 
Powerlessness: 0.69  (D carries!) 

```

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import pickle
from tqdm.auto import tqdm
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report
import torch.nn.functional as F
from itertools import product

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Imports loaded")
print(f"Device: {device}")

Imports loaded
Device: cuda


## 1) Load All 3 Trained Models

In [2]:
print("Loading Model A (Balanced Specialist)...")
model_A = RobertaForSequenceClassification.from_pretrained('models/roberta-ensemble-model-A')
model_A = model_A.to(device)
model_A.eval()
print("  Model A loaded")

print("\nLoading Model D (Rhetoric Specialist - DeBERTa v3)...")
model_D = AutoModelForSequenceClassification.from_pretrained('models/deberta-ensemble-model-D')
model_D = model_D.to(device)
model_D.eval()
print("  Model D loaded")

print("\nLoading Model C (Conservative Baseline)...")
model_C = RobertaForSequenceClassification.from_pretrained('models/roberta-ensemble-model-C')
model_C = model_C.to(device)
model_C.eval()
print("  Model C loaded")

# Load tokenizers and label encoder
tokenizer_A = RobertaTokenizer.from_pretrained('models/roberta-ensemble-model-A')
tokenizer_D = AutoTokenizer.from_pretrained('models/deberta-ensemble-model-D')
tokenizer_C = RobertaTokenizer.from_pretrained('models/roberta-ensemble-model-C')
with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
labels = list(label_encoder.classes_)
num_labels = len(labels)

print(f"\nLabels: {labels}")
print("All models loaded successfully!")

Loading Model A (Balanced Specialist)...


  Model A loaded

Loading Model D (Rhetoric Specialist - DeBERTa v3)...
  Model D loaded

Loading Model C (Conservative Baseline)...
  Model C loaded

Labels: ['Conflict', 'Economic', 'Human Impact', 'Moral Value', 'None', 'Powerlessness']
All models loaded successfully!


## 2) Load Test Data

In [3]:
print("Loading augmented data...")
augmented = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])

val_df = augmented[augmented['split'] == 'validation'].copy()
test_df = augmented[augmented['split'] == 'test'].copy()

val_df['label'] = label_encoder.transform(val_df['frame_label'])
test_df['label'] = label_encoder.transform(test_df['frame_label'])

print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

Loading augmented data...


Validation: 683 samples
Test: 348 samples


## 3) Ensemble Prediction Function

In [4]:
def get_model_predictions(model, tokenizer, texts, batch_size=32, max_length=384):
    """Get logits from a model for given texts using the model's tokenizer"""
    all_logits = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Predicting'):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, truncation=True, max_length=max_length, 
                               padding=True, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())
    return torch.cat(all_logits, dim=0)

def ensemble_probs_class_specific(logits_A, logits_D, logits_C, weights_per_class):
    """Return ensemble probability distribution with class-specific weights."""
    probs_A = F.softmax(logits_A, dim=-1)
    probs_D = F.softmax(logits_D, dim=-1)
    probs_C = F.softmax(logits_C, dim=-1)
    ensemble_probs = torch.zeros_like(probs_A)
    for class_idx in range(num_labels):
        w = weights_per_class[class_idx]
        ensemble_probs[:, class_idx] = (
            w[0] * probs_A[:, class_idx] +
            w[1] * probs_D[:, class_idx] +
            w[2] * probs_C[:, class_idx]
        )
    ensemble_probs = ensemble_probs / ensemble_probs.sum(dim=-1, keepdim=True)
    return ensemble_probs


def ensemble_predict_class_specific(logits_A, logits_D, logits_C, weights_per_class):
    """Ensemble prediction with class-specific weights. Apply weights and take argmax."""
    # Get softmax probabilities
    probs_A = F.softmax(logits_A, dim=-1)  # Shape: (N, num_classes)
    probs_D = F.softmax(logits_D, dim=-1)
    probs_C = F.softmax(logits_C, dim=-1)
    
    # Initialize weighted ensemble
    ensemble_probs = torch.zeros_like(probs_A)
    
    # For each class, apply class-specific weights
    for class_idx in range(len(weights_per_class)):
        w_A, w_D, w_C = weights_per_class[class_idx]
        w_sum = w_A + w_D + w_C
        ensemble_probs[:, class_idx] = (w_A * probs_A[:, class_idx] + 
                                         w_D * probs_D[:, class_idx] + 
                                         w_C * probs_C[:, class_idx]) / w_sum
    
    # Return argmax predictions
    return ensemble_probs.argmax(dim=-1).cpu().numpy()
def ensemble_predict_with_thresholds(logits_A, logits_D, logits_C, weights_per_class, thresholds):
    """Apply per-class thresholds: choose argmax over probs / thresholds."""
    probs = ensemble_probs_class_specific(logits_A, logits_D, logits_C, weights_per_class)
    thr = torch.tensor(thresholds, dtype=probs.dtype).unsqueeze(0).expand_as(probs)
    adjusted = probs / thr
    return adjusted.argmax(dim=-1).numpy()

print("Ensemble functions updated (class-specific weights + per-class thresholds)")

Ensemble functions updated (class-specific weights + per-class thresholds)


## 4) Get Predictions from All Models

In [5]:
print("Getting predictions from Model A...")
val_logits_A = get_model_predictions(model_A, tokenizer_A, val_df['chunk_text'].tolist(), max_length=384)
test_logits_A = get_model_predictions(model_A, tokenizer_A, test_df['chunk_text'].tolist(), max_length=384)

print("\nGetting predictions from Model D (DeBERTa)...")
val_logits_D = get_model_predictions(model_D, tokenizer_D, val_df['chunk_text'].tolist(), max_length=384)
test_logits_D = get_model_predictions(model_D, tokenizer_D, test_df['chunk_text'].tolist(), max_length=384)

print("\nGetting predictions from Model C...")
val_logits_C = get_model_predictions(model_C, tokenizer_C, val_df['chunk_text'].tolist(), max_length=384)
test_logits_C = get_model_predictions(model_C, tokenizer_C, test_df['chunk_text'].tolist(), max_length=384)

print("\nAll predictions obtained!")

Getting predictions from Model A...


Predicting:   0%|          | 0/22 [00:00<?, ?it/s]

Predicting:   0%|          | 0/11 [00:00<?, ?it/s]


Getting predictions from Model D (DeBERTa)...


Predicting:   0%|          | 0/22 [00:00<?, ?it/s]

Predicting:   0%|          | 0/11 [00:00<?, ?it/s]


Getting predictions from Model C...


Predicting:   0%|          | 0/22 [00:00<?, ?it/s]

Predicting:   0%|          | 0/11 [00:00<?, ?it/s]


All predictions obtained!


## 5) Initialize Class-Specific Weights

In [6]:
# Initial weights based on strategy (A, D, C)
# [Model_A_weight, Model_D_weight, Model_C_weight] for each class
weights_per_class_initial = [
    [0.35, 0.40, 0.25],  # Conflict (D may help)
    [0.40, 0.20, 0.40],  # Economic (A & C compensate)
    [0.35, 0.30, 0.35],  # Human Impact (balanced)
    [0.30, 0.50, 0.20],  # Moral Value (D DOMINATES)
    [0.35, 0.30, 0.35],  # None (balanced)
    [0.30, 0.50, 0.20]   # Powerlessness (D DOMINATES)
]

print("Initial class-specific weights (A, D, C):")
for i, label in enumerate(labels):
    w = weights_per_class_initial[i]
    print(f"  {label:20s}: A={w[0]:.2f}, D={w[1]:.2f}, C={w[2]:.2f}")

Initial class-specific weights (A, D, C):
  Conflict            : A=0.35, D=0.40, C=0.25
  Economic            : A=0.40, D=0.20, C=0.40
  Human Impact        : A=0.35, D=0.30, C=0.35
  Moral Value         : A=0.30, D=0.50, C=0.20
  None                : A=0.35, D=0.30, C=0.35
  Powerlessness       : A=0.30, D=0.50, C=0.20


## 6) Evaluate Initial Ensemble

In [7]:
print("Evaluating initial ensemble on validation...")
val_preds_initial = ensemble_predict_class_specific(val_logits_A, val_logits_D, val_logits_C, weights_per_class_initial)
val_f1_initial = f1_score(val_df['label'], val_preds_initial, average='macro')
print(f"\nInitial Validation F1: {val_f1_initial:.4f}")

print("\nPer-class F1 (validation):")
_, _, f1_per_class, _ = precision_recall_fscore_support(
    val_df['label'], val_preds_initial, average=None, zero_division=0
)
for i, label in enumerate(labels):
    print(f"  {label:20s}: {f1_per_class[i]:.4f}")

Evaluating initial ensemble on validation...

Initial Validation F1: 0.7243

Per-class F1 (validation):
  Conflict            : 0.6699
  Economic            : 0.8525
  Human Impact        : 0.7946
  Moral Value         : 0.6348
  None                : 0.7628
  Powerlessness       : 0.6311


## 7) Grid Search for Optimal Weights (Optional)

In [8]:
print("\nOptional: Grid search to fine-tune weights...")
print("Searching around initial weights (+/- 0.1)...")

# For each class, search around initial weights
best_weights = weights_per_class_initial.copy()
best_f1 = val_f1_initial

# Only tune weak classes (Moral Value & Powerlessness)
classes_to_tune = [3, 5]  # Moral Value, Powerlessness

for class_idx in classes_to_tune:
    print(f"\nTuning {labels[class_idx]}...")
    
    # Try different B weights (0.4, 0.5, 0.6)
    for b_weight in [0.40, 0.50, 0.60]:
        # Distribute remaining weight between A and C
        remaining = 1.0 - b_weight
        for a_ratio in [0.3, 0.4, 0.5, 0.6, 0.7]:
            a_weight = remaining * a_ratio
            c_weight = remaining * (1 - a_ratio)
            
            # Test this configuration
            test_weights = [w.copy() for w in best_weights]
            test_weights[class_idx] = [a_weight, b_weight, c_weight]
            
            preds = ensemble_predict_class_specific(val_logits_A, val_logits_D, val_logits_C, test_weights)
            f1 = f1_score(val_df['label'], preds, average='macro')
            
            if f1 > best_f1:
                best_f1 = f1
                best_weights = test_weights
                print(f"  New best: A={a_weight:.2f}, B={b_weight:.2f}, C={c_weight:.2f} -> F1={f1:.4f}")

print(f"\nBest Validation F1: {best_f1:.4f}")
print("\nOptimized weights:")
for i, label in enumerate(labels):
    w = best_weights[i]
    print(f"  {label:20s}: A={w[0]:.2f}, B={w[1]:.2f}, C={w[2]:.2f}")


Optional: Grid search to fine-tune weights...
Searching around initial weights (+/- 0.1)...

Tuning Moral Value...
  New best: A=0.18, B=0.40, C=0.42 -> F1=0.7304

Tuning Powerlessness...
  New best: A=0.25, B=0.50, C=0.25 -> F1=0.7307
  New best: A=0.20, B=0.60, C=0.20 -> F1=0.7322

Best Validation F1: 0.7322

Optimized weights:
  Conflict            : A=0.35, B=0.40, C=0.25
  Economic            : A=0.40, B=0.20, C=0.40
  Human Impact        : A=0.35, B=0.30, C=0.35
  Moral Value         : A=0.18, B=0.40, C=0.42
  None                : A=0.35, B=0.30, C=0.35
  Powerlessness       : A=0.20, B=0.60, C=0.20


## 8) Final Test Evaluation

In [9]:
print("\n" + "="*80)
print("FINAL ENSEMBLE EVALUATION ON TEST SET")
print("="*80)

test_preds_final = ensemble_predict_class_specific(test_logits_A, test_logits_D, test_logits_C, best_weights)

# Overall metrics
test_acc = accuracy_score(test_df['label'], test_preds_final)
test_f1_macro = f1_score(test_df['label'], test_preds_final, average='macro')
test_f1_weighted = f1_score(test_df['label'], test_preds_final, average='weighted')

print(f"\nTest Accuracy:  {test_acc:.4f}")
print(f"Test F1 Macro:  {test_f1_macro:.4f} ⭐")
print(f"Test F1 Weighted: {test_f1_weighted:.4f}")

# Per-class F1
print("\nPer-Class F1 Scores:")
_, _, f1_per_class_test, _ = precision_recall_fscore_support(
    test_df['label'], test_preds_final, average=None, zero_division=0
)
for i, label in enumerate(labels):
    print(f"  {label:20s}: {f1_per_class_test[i]:.4f}")

# Full classification report
print("\n" + "="*80)
print("CLASSIFICATION REPORT:")
print("="*80)
print(classification_report(test_df['label'], test_preds_final, 
                          target_names=labels, digits=4))


FINAL ENSEMBLE EVALUATION ON TEST SET

Test Accuracy:  0.6925
Test F1 Macro:  0.6952 ⭐
Test F1 Weighted: 0.6952

Per-Class F1 Scores:
  Conflict            : 0.7010
  Economic            : 0.7544
  Human Impact        : 0.7179
  Moral Value         : 0.6331
  None                : 0.7788
  Powerlessness       : 0.5862

CLASSIFICATION REPORT:
               precision    recall  f1-score   support

     Conflict     0.8718    0.5862    0.7010        58
     Economic     0.7679    0.7414    0.7544        58
 Human Impact     0.7119    0.7241    0.7179        58
  Moral Value     0.5432    0.7586    0.6331        58
         None     0.8000    0.7586    0.7788        58
Powerlessness     0.5862    0.5862    0.5862        58

     accuracy                         0.6925       348
    macro avg     0.7135    0.6925    0.6952       348
 weighted avg     0.7135    0.6925    0.6952       348



## 9) Save Ensemble Results

In [10]:
# Save ensemble configuration and results
os.makedirs('results', exist_ok=True)
with open('results/ensemble_final_results.json', 'w') as f:
    json.dump({
        'ensemble_strategy': 'Class-specific weighted fusion',
        'models': ['A - Balanced', 'B - Weak Champion', 'C - Conservative'],
        'weights_per_class': {labels[i]: best_weights[i] for i in range(num_labels)},
        'test_metrics': {
            'accuracy': float(test_acc),
            'f1_macro': float(test_f1_macro),
            'f1_weighted': float(test_f1_weighted),
            'f1_per_class': {labels[i]: float(f1_per_class_test[i]) for i in range(num_labels)}
        },
        'validation_f1': float(best_f1)
    }, f, indent=2)

print("\nResults saved to results/ensemble_final_results.json")

print("\n" + "="*80)
print("ENSEMBLE COMPLETE!")
print("="*80)
print(f"Final Test F1: {test_f1_macro:.4f}")
if test_f1_macro >= 0.75:
    print("TARGET ACHIEVED: F1 >= 0.75! ⭐⭐⭐")
elif test_f1_macro >= 0.73:
    print("Excellent result! Very close to 0.75 target.")
else:
    print(f"Gap to target: {0.75 - test_f1_macro:.4f}")
print("="*80)


Results saved to results/ensemble_final_results.json

ENSEMBLE COMPLETE!
Final Test F1: 0.6952
Gap to target: 0.0548


## 7b) Per-class Threshold Grid Search (validation)

In [11]:
print('Grid searching per-class thresholds...')
threshold_grid = {
    'default': [1.0],
    'Moral Value': [0.85, 0.90, 0.95, 1.0],
    'Powerlessness': [0.85, 0.90, 0.95, 1.0]
}
from itertools import product
def grid_search_thresholds(logits_A, logits_D, logits_C, weights_per_class, labels_true):
    best_thr = [1.0]*num_labels; best_f1 = 0.0
    spaces = [threshold_grid.get(lbl, threshold_grid['default']) for lbl in labels]
    for combo in product(*spaces):
        preds = ensemble_predict_with_thresholds(logits_A, logits_D, logits_C, weights_per_class, combo)
        f1 = f1_score(labels_true, preds, average='macro')
        if f1 > best_f1: best_f1, best_thr = f1, list(combo)
    return best_thr, best_f1
best_thresholds, f1_thr = grid_search_thresholds(val_logits_A, val_logits_D, val_logits_C, best_weights, val_df['label'].to_numpy())
print('Best thresholds:', best_thresholds)
print(f'Validation F1 with thresholds: {f1_thr:.4f}')


Grid searching per-class thresholds...
Best thresholds: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Validation F1 with thresholds: 0.7322


## 8b) Final Test Evaluation with Thresholds

In [12]:
test_preds_final_thr = ensemble_predict_with_thresholds(test_logits_A, test_logits_D, test_logits_C, best_weights, best_thresholds)
test_acc_thr = accuracy_score(test_df['label'], test_preds_final_thr)
test_f1_macro_thr = f1_score(test_df['label'], test_preds_final_thr, average='macro')
print('Test F1 Macro (thresholded):', round(test_f1_macro_thr,4))


Test F1 Macro (thresholded): 0.6952
